# Depression Detection — Exploratory Data Analysis

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from data import load_reddit_dataset, preprocess
from features import linguistic_features

sns.set_theme(style='whitegrid')

## 1. Load & inspect

In [ ]:
df_raw = load_reddit_dataset()
print(df_raw.shape)
df_raw.head()

In [ ]:
df = preprocess(df_raw)
df['label'].value_counts()

## 2. Class balance

In [ ]:
ax = df['label'].value_counts().plot(kind='bar', color=['steelblue', 'salmon'])
ax.set_title('Class distribution')
ax.set_xlabel('Label')
ax.set_ylabel('Count')
plt.tight_layout()

## 3. Text length distribution

In [ ]:
df['text_len'] = df['text'].str.split().str.len()
fig, ax = plt.subplots()
for label, grp in df.groupby('label'):
    grp['text_len'].clip(upper=300).plot.hist(bins=50, alpha=0.6, label=label, ax=ax)
ax.set_title('Post length by class')
ax.set_xlabel('Word count')
ax.legend()
plt.tight_layout()

## 4. Word clouds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, label in zip(axes, df['label'].unique()):
    corpus = ' '.join(df[df['label'] == label]['text'])
    wc = WordCloud(width=600, height=400, background_color='white').generate(corpus)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(label)
plt.tight_layout()

## 5. Linguistic features comparison

In [ ]:
import numpy as np
feat_names = ['log_length', 'first_person_rate', 'negation_rate', 'neg_emotion_rate', 'avg_word_len']
feats = linguistic_features(df['text'].tolist())
feat_df = pd.DataFrame(feats, columns=feat_names)
feat_df['label'] = df['label'].values
feat_df.groupby('label')[feat_names].mean().T.plot(kind='bar')
plt.title('Mean linguistic features by class')
plt.xticks(rotation=30)
plt.tight_layout()